In [1]:
# ================================================================
# CABITZA ET AL. COVID-19 MODEL REPRODUCTION
# INTERNAL-EXTERNAL VALIDATION
# LOGISTIC REGRESSION + RANDOM FOREST
# ================================================================

from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    roc_auc_score,
    brier_score_loss
)

from imblearn.over_sampling import SMOTE

warnings.filterwarnings("ignore")

# ================================================================
# 1. SETTINGS
# ================================================================

RANDOM_STATE = 42
N_REPEATS = 100

PROJECT_ROOT = Path(
    r"C:\Users\bl\Documents\ML-Projects\covid-laboratory-prediction"
)

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "all_training.csv"
RESULTS_DIR = PROJECT_ROOT / "results" / "reproduction"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 75)
print("CABITZA ET AL. — INTERNAL-EXTERNAL VALIDATION")
print("LOGISTIC REGRESSION + RANDOM FOREST")
print("=" * 75)

print("Dataset exists:", DATA_PATH.exists())


# ================================================================
# 2. LOAD DATA
# ================================================================

df = pd.read_csv(DATA_PATH)

id_col = "Unnamed: 0"
target_col = "target"

ids = df[id_col].astype(str).str.strip()


def assign_cohort(patient_id):

    if patient_id.startswith("A"):
        return "training"

    elif patient_id.startswith("PSMAY"):
        return "training"

    elif patient_id.isdigit() and 2001 <= int(patient_id) <= 2058:
        return "internal_external"

    elif patient_id.isdigit():
        return "external"

    else:
        return "unclassified"


df["cohort"] = ids.apply(assign_cohort)

print("\nReconstructed cohorts:")
print(df["cohort"].value_counts())


# ================================================================
# 3. DEFINE PREDICTORS
# ================================================================

predictors = [
    col for col in df.columns
    if col not in [id_col, target_col, "cohort"]
]

print("\nNumber of predictors:", len(predictors))


# ================================================================
# 4. OSR DEVELOPMENT COHORT
# ================================================================

osr = df[
    df["cohort"] == "training"
].copy()

X_osr = osr[predictors].copy()
y_osr = osr[target_col].astype(int).copy()

print("\nOSR development cohort:")
print("N =", len(osr))
print(y_osr.value_counts().sort_index())


# ================================================================
# 5. IOG INTERNAL-EXTERNAL COHORT
# ================================================================

iog = df[
    df["cohort"] == "internal_external"
].copy()

X_iog = iog[predictors].copy()
y_iog = iog[target_col].astype(int).copy()

print("\nIOG internal-external cohort:")
print("N =", len(iog))
print(y_iog.value_counts().sort_index())

print(
    "\nNOTE: Public CSV reconstructs IOG as "
    f"{(y_iog == 0).sum()} negative / "
    f"{(y_iog == 1).sum()} positive."
)


# ================================================================
# 6. MODEL CONFIGURATIONS
#
# These use the best configurations identified during our
# reproduction of the OSR development models.
# ================================================================

# ---------------- LOGISTIC REGRESSION ----------------

lr_selector = LogisticRegression(
    C=0.1,
    penalty="l1",
    solver="liblinear",
    max_iter=5000,
    random_state=RANDOM_STATE
)

lr_classifier = LogisticRegression(
    C=0.1,
    penalty="l1",
    solver="liblinear",
    max_iter=5000,
    random_state=RANDOM_STATE
)

lr_pipeline = Pipeline([
    (
        "imputer",
        KNNImputer(n_neighbors=5)
    ),
    (
        "scaler",
        StandardScaler()
    ),
    (
        "rfe",
        RFE(
            estimator=lr_selector,
            n_features_to_select=34,
            step=1
        )
    ),
    (
        "classifier",
        lr_classifier
    )
])


# ---------------- RANDOM FOREST ----------------

rf_selector = RandomForestClassifier(
    n_estimators=100,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_classifier = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=5,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_pipeline = Pipeline([
    (
        "imputer",
        KNNImputer(n_neighbors=5)
    ),
    (
        "rfe",
        RFE(
            estimator=rf_selector,
            n_features_to_select=34,
            step=1
        )
    ),
    (
        "classifier",
        rf_classifier
    )
])


# ================================================================
# 7. HELPER FUNCTION FOR PERFORMANCE
# ================================================================

def calculate_metrics(y_true, y_pred, y_prob):

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else np.nan
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    # AUC requires both outcome classes
    if len(np.unique(y_true)) == 2:
        auc = roc_auc_score(
            y_true,
            y_prob
        )
    else:
        auc = np.nan

    brier = brier_score_loss(
        y_true,
        y_prob
    )

    return {
        "auc": auc,
        "accuracy": accuracy,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "brier_score": brier,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }


# ================================================================
# 8. INTERNAL-EXTERNAL VALIDATION
#
# Reported Cabitza strategy:
#
# - 100 random 50/50 IOG train/test splits
# - SMOTE applied to IOG training half
# - Generate a 1,624-case IOG-derived training dataset
# - Combine with 1,624 OSR cases
# - Final training N = 3,248
# - Evaluate on untouched IOG test half
#
# IMPORTANT:
# Missing values must be imputed before SMOTE because SMOTE
# cannot operate on NaN values.
#
# We therefore fit the KNN imputer using the IOG training
# subset only before SMOTE generation.
# ================================================================

lr_runs = []
rf_runs = []

print("\n" + "=" * 75)
print("STARTING 100 INTERNAL-EXTERNAL VALIDATION REPEATS")
print("=" * 75)

start_time = time.time()

for repeat in range(N_REPEATS):

    seed = RANDOM_STATE + repeat

    # ------------------------------------------------------------
    # Stratified 50:50 split of IOG cohort
    # ------------------------------------------------------------

    (
        X_iog_train,
        X_iog_test,
        y_iog_train,
        y_iog_test
    ) = train_test_split(
        X_iog,
        y_iog,
        test_size=0.50,
        stratify=y_iog,
        random_state=seed
    )

    # ------------------------------------------------------------
    # Impute IOG training data specifically for SMOTE generation
    #
    # This imputer sees ONLY the IOG training half.
    # ------------------------------------------------------------

    smote_imputer = KNNImputer(
        n_neighbors=5
    )

    X_iog_train_imputed = smote_imputer.fit_transform(
        X_iog_train
    )

    # ------------------------------------------------------------
    # SMOTE TO 1,624 TOTAL IOG-DERIVED INSTANCES
    #
    # We target 812 cases per class:
    # 812 + 812 = 1,624.
    # ------------------------------------------------------------

    smote = SMOTE(
        sampling_strategy={
            0: 812,
            1: 812
        },
        random_state=seed,
        k_neighbors=5
    )

    X_iog_smote, y_iog_smote = smote.fit_resample(
        X_iog_train_imputed,
        y_iog_train
    )

    # Convert synthetic data back to DataFrame
    X_iog_smote = pd.DataFrame(
        X_iog_smote,
        columns=predictors
    )

    y_iog_smote = pd.Series(
        y_iog_smote,
        name=target_col
    )

    # ------------------------------------------------------------
    # IMPORTANT:
    # OSR retains its original raw values/missingness.
    #
    # The synthetic IOG sample is already imputed because
    # SMOTE cannot work with missing data.
    #
    # Both are then passed through the model pipeline.
    # ------------------------------------------------------------

    X_combined = pd.concat(
        [
            X_osr.reset_index(drop=True),
            X_iog_smote.reset_index(drop=True)
        ],
        ignore_index=True
    )

    y_combined = pd.concat(
        [
            y_osr.reset_index(drop=True),
            y_iog_smote.reset_index(drop=True)
        ],
        ignore_index=True
    )

    # Safety check
    assert len(X_combined) == 3248
    assert len(y_combined) == 3248

    # ============================================================
    # LOGISTIC REGRESSION
    # ============================================================

    lr_model = clone(
        lr_pipeline
    )

    lr_model.fit(
        X_combined,
        y_combined
    )

    lr_pred = lr_model.predict(
        X_iog_test
    )

    lr_prob = lr_model.predict_proba(
        X_iog_test
    )[:, 1]

    lr_metrics = calculate_metrics(
        y_iog_test,
        lr_pred,
        lr_prob
    )

    lr_metrics.update({
        "repeat": repeat + 1,
        "seed": seed,
        "n_train": len(X_combined),
        "n_test": len(X_iog_test)
    })

    lr_runs.append(
        lr_metrics
    )

    # ============================================================
    # RANDOM FOREST
    # ============================================================

    rf_model = clone(
        rf_pipeline
    )

    rf_model.fit(
        X_combined,
        y_combined
    )

    rf_pred = rf_model.predict(
        X_iog_test
    )

    rf_prob = rf_model.predict_proba(
        X_iog_test
    )[:, 1]

    rf_metrics = calculate_metrics(
        y_iog_test,
        rf_pred,
        rf_prob
    )

    rf_metrics.update({
        "repeat": repeat + 1,
        "seed": seed,
        "n_train": len(X_combined),
        "n_test": len(X_iog_test)
    })

    rf_runs.append(
        rf_metrics
    )

    # Progress report
    if (repeat + 1) % 10 == 0:
        print(
            f"Completed {repeat + 1}/{N_REPEATS} repeats"
        )


# ================================================================
# 9. CONVERT RUN RESULTS TO DATAFRAMES
# ================================================================

lr_runs_df = pd.DataFrame(
    lr_runs
)

rf_runs_df = pd.DataFrame(
    rf_runs
)

lr_runs_df["model"] = "Logistic Regression"
rf_runs_df["model"] = "Random Forest"


# ================================================================
# 10. SUMMARY FUNCTION
# ================================================================

def summarize_validation(df, model_name):

    metrics = [
        "auc",
        "accuracy",
        "sensitivity",
        "specificity",
        "brier_score"
    ]

    rows = []

    for metric in metrics:

        rows.append({
            "model": model_name,
            "metric": metric,
            "mean": df[metric].mean(),
            "sd": df[metric].std(),
            "median": df[metric].median(),
            "min": df[metric].min(),
            "max": df[metric].max()
        })

    return pd.DataFrame(rows)


lr_summary = summarize_validation(
    lr_runs_df,
    "Logistic Regression"
)

rf_summary = summarize_validation(
    rf_runs_df,
    "Random Forest"
)

validation_summary = pd.concat(
    [
        lr_summary,
        rf_summary
    ],
    ignore_index=True
)


# ================================================================
# 11. DISPLAY RESULTS
# ================================================================

elapsed = time.time() - start_time

print("\n" + "=" * 75)
print("INTERNAL-EXTERNAL VALIDATION COMPLETE")
print("=" * 75)

print(
    f"Elapsed time: {elapsed / 60:.2f} minutes"
)

print("\nLOGISTIC REGRESSION — 100-repeat mean performance")

for metric in [
    "auc",
    "accuracy",
    "sensitivity",
    "specificity",
    "brier_score"
]:
    print(
        f"{metric:12s}: "
        f"{lr_runs_df[metric].mean():.4f} "
        f"± {lr_runs_df[metric].std():.4f}"
    )


print("\nRANDOM FOREST — 100-repeat mean performance")

for metric in [
    "auc",
    "accuracy",
    "sensitivity",
    "specificity",
    "brier_score"
]:
    print(
        f"{metric:12s}: "
        f"{rf_runs_df[metric].mean():.4f} "
        f"± {rf_runs_df[metric].std():.4f}"
    )


print("\nFull summary:")
print(
    validation_summary.round(4).to_string(
        index=False
    )
)


# ================================================================
# 12. SAVE RESULTS
# ================================================================

lr_runs_df.to_csv(
    RESULTS_DIR /
    "lr_internal_external_validation_runs.csv",
    index=False
)

rf_runs_df.to_csv(
    RESULTS_DIR /
    "rf_internal_external_validation_runs.csv",
    index=False
)

validation_summary.to_csv(
    RESULTS_DIR /
    "lr_rf_internal_external_validation_summary.csv",
    index=False
)


print("\nSaved:")
print(
    RESULTS_DIR /
    "lr_internal_external_validation_runs.csv"
)

print(
    RESULTS_DIR /
    "rf_internal_external_validation_runs.csv"
)

print(
    RESULTS_DIR /
    "lr_rf_internal_external_validation_summary.csv"
)

print("\nInternal-external validation finished.")

CABITZA ET AL. — INTERNAL-EXTERNAL VALIDATION
LOGISTIC REGRESSION + RANDOM FOREST
Dataset exists: True

Reconstructed cohorts:
cohort
training             1624
internal_external      58
external               54
Name: count, dtype: int64

Number of predictors: 34

OSR development cohort:
N = 1624
target
0    838
1    786
Name: count, dtype: int64

IOG internal-external cohort:
N = 58
target
0    28
1    30
Name: count, dtype: int64

NOTE: Public CSV reconstructs IOG as 28 negative / 30 positive.

STARTING 100 INTERNAL-EXTERNAL VALIDATION REPEATS
Completed 10/100 repeats
Completed 20/100 repeats
Completed 30/100 repeats
Completed 40/100 repeats
Completed 50/100 repeats
Completed 60/100 repeats
Completed 70/100 repeats
Completed 80/100 repeats
Completed 90/100 repeats
Completed 100/100 repeats

INTERNAL-EXTERNAL VALIDATION COMPLETE
Elapsed time: 8.32 minutes

LOGISTIC REGRESSION — 100-repeat mean performance
auc         : 0.8949 ± 0.0509
accuracy    : 0.8107 ± 0.0640
sensitivity : 0.7027